## Import

In [ ]:
import warnings
import numpy as np
from category_encoders import MEstimateEncoder
from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import mutual_info_regression
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
from pathlib import Path
warnings.filterwarnings("ignore")

## Import Data 

In [ ]:


csv_candidates = [
    Path("student_dataset/student_failure/train.csv"),
    Path("../student_dataset/student_failure/train.csv"),
]

for csv_path in csv_candidates:
    if csv_path.exists():
        data = pd.read_csv(csv_path)
        break


# Afficher les premières lignes du dataset
X = data.drop(["score_examen", "id"], axis=1)
y = data.score_examen
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.8, test_size=0.2, random_state=0
)

In [ ]:
X.head()
y_train.head()

## Data Exploration and Analysis

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(y_train, kde=True, color='skyblue')
plt.axvline(50, color='red', linestyle='--', label="Seuil d'échec")
plt.title("Distribution des notes (Train Set)")
plt.legend()
plt.savefig("../Graph/score_distribution.png")
plt.show()


In [ ]:
X_mi = X_train.copy()

for col in X_mi.select_dtypes(include='number').columns:
    X_mi[col] = X_mi[col].fillna(X_mi[col].median())
 
for col in X_mi.select_dtypes(include='object').columns:
    X_mi[col], _ = X_mi[col].factorize()

discrete_features = X_mi.dtypes == int

mi_scores = mutual_info_regression(X_mi, y_train, discrete_features=discrete_features, random_state=0)
mi_scores_series = pd.Series(mi_scores, index=X_train.columns).sort_values(ascending=False)
plt.figure(figsize=(10, 6))
sns.barplot(x=mi_scores_series.values, y=mi_scores_series.index, palette='viridis')
plt.title("Importance des variables (Mutual Information)")
plt.xlabel("Score MI")
plt.savefig("../Graph/mi_scores.png")
plt.show()


In [ ]:
print("Mutual Information Scores:", mi_scores_series)

In [ ]:
# --- 1. Analyse de la variance expliquée (PCA sans restriction) ---

def plot_variance(pca, width=8, dpi=100):
    fig, axs = plt.subplots(1, 2)
    n = pca.n_components_
    grid = np.arange(1, n + 1)
    evr = pca.explained_variance_ratio_
    axs[0].bar(grid, evr)
    axs[0].set(xlabel="Composante", title="% Variance Expliquée", ylim=(0.0, 1.0))
    cv = np.cumsum(evr)
    axs[1].plot(np.r_[0, grid], np.r_[0, cv], "o-")
    axs[1].set(xlabel="Composante", title="% Variance Cumulée", ylim=(0.0, 1.0))
    fig.set(figwidth=width, dpi=dpi)
    plt.tight_layout()
    return axs

# Garder uniquement les features numériques CONTINUES pour la PCA.
# Les colonnes catégorielles encodées par factorize() donnent des codes
# arbitraires (0, 1, 2...) sans relation de distance : la PCA les interpréterait
# comme des quantités continues, ce qui fausse les composantes.
numeric_features = X_train.select_dtypes(include='number').columns.tolist()
X_numeric = X_mi[numeric_features]
feature_names = X_numeric.columns  # utilisé dans toutes les cellules PCA suivantes

print(f"Features numériques utilisées pour la PCA ({len(numeric_features)}) :")
print(numeric_features)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_numeric)

# PCA sans restriction pour voir toutes les composantes
pca_full = PCA()
pca_full.fit(X_scaled)

plot_variance(pca_full)
plt.savefig("../Graph/pca_variance.png")
plt.show()

print("\nVariance expliquée par composante :")
for i, evr in enumerate(pca_full.explained_variance_ratio_):
    print(f"  PC{i+1}: {evr*100:.1f}%  (cumulé: {np.cumsum(pca_full.explained_variance_ratio_)[i]*100:.1f}%)")

In [ ]:
# --- 2. MI Scores des composantes PCA ---
# Quelles composantes sont les plus prédictives de score_examen ?

X_pca_full = pca_full.transform(X_scaled)
component_names = [f"PC{i+1}" for i in range(X_pca_full.shape[1])]
X_pca_df = pd.DataFrame(X_pca_full, columns=component_names, index=X_mi.index)

mi_scores_pca = mutual_info_regression(X_pca_df, y_train, discrete_features=False, random_state=0)
mi_scores_pca = pd.Series(mi_scores_pca, name="MI Score", index=component_names).sort_values(ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=mi_scores_pca.values, y=mi_scores_pca.index, palette="viridis")
plt.title("MI Scores des composantes PCA (vs score_examen)")
plt.xlabel("Score MI")
plt.tight_layout()
plt.savefig("../Graph/pca_mi_scores.png")
plt.show()

print(mi_scores_pca)

In [ ]:
# --- 3. Analyse des loadings et feature engineering ---

# feature_names est défini dans la cellule PCA (uniquement les colonnes numériques)
loadings = pd.DataFrame(
    pca_full.components_.T,
    columns=component_names,
    index=feature_names
)

# Sélectionner uniquement les composantes qui atteignent 95% de variance cumulée.
# Hardcoder à 4 était arbitraire et cachait des composantes potentiellement
# informatives révélées par les MI scores ci-dessus.
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n_sig = int(np.searchsorted(cumvar, 0.95)) + 1
n_sig = min(n_sig, len(component_names))

print(f"Affichage des {n_sig} composantes significatives (>= 95% variance cumulée)")
display(
    loadings.iloc[:, :n_sig]
    .style.background_gradient(cmap="coolwarm", axis=None)
    .format("{:.3f}")
)

# Identifier le contraste dominant dans PC1 (feature la plus positive vs la plus négative)
pc1_loadings = loadings["PC1"].sort_values()
top_negative = pc1_loadings.head(1).index[0]
top_positive = pc1_loadings.tail(1).index[0]

print(f"\nContraste PC1 : '{top_positive}' (positif) vs '{top_negative}' (négatif)")

# Créer une feature ratio inspirée du contraste PC1
ratio_col = f"ratio_{top_positive}_vs_{top_negative}"
X_mi[ratio_col] = X_mi[top_positive] / (X_mi[top_negative].abs() + 1e-6)

print(f"Nouvelle feature créée : {ratio_col}")
print(X_mi[ratio_col].describe())

# Vérifier l'utilité de cette feature avec MI
mi_ratio = mutual_info_regression(
    X_mi[[ratio_col]], y_train, discrete_features=False, random_state=0
)
print(f"\nMI Score de '{ratio_col}' : {mi_ratio[0]:.4f}")

In [ ]:
# --- 4. Vrai Biplot PCA (scatter + flèches des loadings) ---
# Refit PCA à 2 composantes pour la visualisation 2D

pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_scaled)

loadings_2d = pd.DataFrame(
    pca_2d.components_.T,
    columns=["PC1", "PC2"],
    index=feature_names
)

def get_axis_description(ldf, pc_name, top_n=2):
    top_features = ldf[pc_name].abs().sort_values(ascending=False).head(top_n).index.tolist()
    return " + ".join(top_features)

label_pc1 = f"PC1 : {get_axis_description(loadings_2d, 'PC1')} ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)"
label_pc2 = f"PC2 : {get_axis_description(loadings_2d, 'PC2')} ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)"

status = (y_train < 50).map({
    False: "Réussite (Score > 50)",
    True: "Échec (Tutorat requis)"
})
status_palette = {
    "Réussite (Score > 50)": "#8fb3e8",
    "Échec (Tutorat requis)": "#efb79f"
}

fig, ax = plt.subplots(figsize=(12, 8))

sns.scatterplot(
    x=X_pca_2d[:, 0],
    y=X_pca_2d[:, 1],
    hue=status,
    hue_order=["Réussite (Score > 50)", "Échec (Tutorat requis)"],
    palette=status_palette,
    alpha=0.5,
    edgecolor=None,
    ax=ax
)

# Flèches des loadings (ce qui rend le biplot "vrai")
scale = 3
for feature in feature_names:
    fx = loadings_2d.loc[feature, "PC1"] * scale
    fy = loadings_2d.loc[feature, "PC2"] * scale
    ax.annotate(
        "",
        xy=(fx, fy),
        xytext=(0, 0),
        arrowprops=dict(arrowstyle="->", color="red", lw=1.5)
    )
    ax.text(fx * 1.1, fy * 1.1, feature, fontsize=8, color="darkred", fontweight="bold")

ax.axvline(0, color="black", linestyle="-", alpha=0.2)
ax.axhline(0, color="black", linestyle="-", alpha=0.2)
ax.set_title("PCA Biplot : Profils étudiants et facteurs d'influence", fontsize=15, pad=20)
ax.set_xlabel(label_pc1, fontsize=12, fontweight="bold")
ax.set_ylabel(label_pc2, fontsize=12, fontweight="bold")
ax.legend(title="Statut", loc="upper right")
ax.grid(True, linestyle=":", alpha=0.4)

plt.tight_layout()
plt.savefig("../Graph/pca_biplot.png")
plt.show()

### Clustering with K-MEANS


## Data encoding et scaling 


In [ ]:
ratio_col = "ratio_heures_etude_vs_heures_fête"
for df in (X_train, X_test):
    # remplir les NaN des colonnes sources si nécessaire
    for c in ("heures_etude", "heures_fête"):
        if c in df.columns:
            df[c] = df[c].fillna(X_train[c].median())
# calculer le ratio (évite division par 0)
X_train[ratio_col] = X_train["heures_etude"] / (X_train["heures_fête"].abs() + 1e-6)
X_test[ratio_col]  = X_test["heures_etude"]  / (X_test["heures_fête"].abs()  + 1e-6)

#selection of features based on mutual information scores
features = ["heures_etude","genre","assiduité_classe","diplôme","accès_internet","heures_sommeil","méthode_etude","qualité_sommeil","évaluation_établissement","age","difficulté_examen","ratio_heures_etude_vs_heures_fête"]
X_train_selected = X_train[features]
X_test_selected = X_test[features]




In [ ]:
# check for missing values in the selected features
cols_with_missing = [col for col in X_train_selected.columns
                     if X_train_selected[col].isnull().any()]
print("Colonnes avec valeurs manquantes :", cols_with_missing)
# nombre de valeurs manquantes par colonne
missing_counts = X_train_selected[cols_with_missing].isnull().sum()
print("\nNombre de valeurs manquantes par colonne :")
print(missing_counts)
# pourcentage de valeurs manquantes par colonne
missing_percent = (missing_counts / len(X_train_selected)) * 100
print("\nPourcentage de valeurs manquantes par colonne :")
print(missing_percent)
# imputation des valeurs manquantes
X_train_plus = X_train_selected.copy()
X_test_plus = X_test_selected.copy()
for col in cols_with_missing:
    X_train_plus[col + '_was_missing'] = X_train_plus[col].isnull()
    X_test_plus[col + '_was_missing'] = X_test_plus[col].isnull()
X_train_plus_imputed = X_train_plus.copy()
X_test_plus_imputed  = X_test_plus.copy()
cat_cols = X_train_plus.select_dtypes(include=['object','category']).columns.tolist()
num_cols = X_train_plus.select_dtypes(include=[np.number]).columns.tolist()
if num_cols:
    num_imp = SimpleImputer(strategy='median')
    X_train_plus_imputed[num_cols] = pd.DataFrame(num_imp.fit_transform(X_train_plus[num_cols]),
                                          columns=num_cols, index=X_train_plus.index)
    X_test_plus_imputed[num_cols]  = pd.DataFrame(num_imp.transform(X_test_plus[num_cols]),
                                          columns=num_cols, index=X_test_plus.index)

if cat_cols:
    # stratégie sûre : most_frequent ou constante 'Missing'
    cat_imp = SimpleImputer(strategy='most_frequent')
    X_train_plus_imputed[cat_cols] = pd.DataFrame(cat_imp.fit_transform(X_train_plus[cat_cols]),
                                          columns=cat_cols, index=X_train_plus.index)
    X_test_plus_imputed[cat_cols]  = pd.DataFrame(cat_imp.transform(X_test_plus[cat_cols]),
                                          columns=cat_cols, index=X_test_plus.index)

X_train_plus_imputed.columns = X_train_plus.columns
X_test_plus_imputed.columns = X_test_plus.columns
X_train_plus_imputed.head()


In [ ]:

# encodage des variables catégorielles
categorical_cols = X_train_plus_imputed.select_dtypes(include='object').columns.tolist()
# nombre de modalités par variable catégorielle
for col in categorical_cols:
    n_modalities = X_train_plus_imputed[col].nunique()
    print(f"Variable '{col}' : {n_modalities} modalités")

ordinal_features = ["genre", "accès_internet", "évaluation_établissement", "difficulté_examen", "qualité_sommeil"]
one_hot_features = ["diplôme", "méthode_etude"]
label_X_train = X_train_plus_imputed.copy()
label_X_valid = X_test_plus_imputed.copy()
ordinal_encoder = OrdinalEncoder()
label_X_train[ordinal_features] = ordinal_encoder.fit_transform(X_train_plus_imputed[ordinal_features])
label_X_valid[ordinal_features] = ordinal_encoder.transform(X_test_plus_imputed[ordinal_features])

# one-hot encoding pour les variables à plus de 3 modalités
OH_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
OH_cols_train = pd.DataFrame(
    OH_encoder.fit_transform(label_X_train[one_hot_features]),
    columns=OH_encoder.get_feature_names_out(one_hot_features),
    index=label_X_train.index
)
OH_cols_valid = pd.DataFrame(
    OH_encoder.transform(label_X_valid[one_hot_features]),
    columns=OH_encoder.get_feature_names_out(one_hot_features),
    index=label_X_valid.index
)

# Remove categorical columns (will replace with one-hot encoding)
num_X_train = label_X_train.drop(one_hot_features, axis=1)
num_X_valid = label_X_valid.drop(one_hot_features, axis=1)

# Add one-hot encoded columns to numerical features
OH_X_train = pd.concat([num_X_train, OH_cols_train], axis=1)
OH_X_valid = pd.concat([num_X_valid, OH_cols_valid], axis=1)

OH_X_train.head()


## First model using randomforest


In [ ]:
# utilisation de random forest 
Model_1 = RandomForestRegressor(random_state=1)
Model_1.fit(OH_X_train, y_train)
preds_1 = Model_1.predict(OH_X_valid)
mae_1 = mean_absolute_error(y_test, preds_1)
print(f"MAE du modèle 1 (Random Forest) : {mae_1:.4f}")
naive_mae = mean_absolute_error(y_test, [y_train.mean()] * len(y_test))
print(f"MAE naïf : {naive_mae:.4f}")

## Second model using XGBOOST

In [ ]:

# utilisation de XGBOOST
Model_2 = XGBRegressor(random_state=1, n_estimators=1000, learning_rate=0.05, n_jobs=4,
                       early_stopping_rounds=5)
Model_2.fit(OH_X_train, y_train,
            eval_set=[(OH_X_valid, y_test)],
            verbose=False
            )
preds_2 = Model_2.predict(OH_X_valid)
mae_2 = mean_absolute_error(y_test, preds_2)
print(f"MAE du modèle 2 (XGBoost) : {mae_2:.4f}")
